In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import glob
import json
import datetime
import statistics
from uuid import uuid4

import yaml
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, HnswConfigDiff
from fastembed import TextEmbedding
from chonkie import RecursiveChunker
import chonkie

from src.evaluator import Evaluator
from src.all_functionality import async_chat_wrapper

Evaluation functions defined.
Visualization function defined.


### Full notion

In [2]:
from pathlib import Path

docs_dir = Path("./rag_exp/notion_docs")
md_paths = sorted(docs_dir.glob("**/*.md"))

notion_docs = {}
for p in md_paths:
    with p.open("r", encoding="utf-8") as fh:
        notion_docs[p.stem] = fh.read()

print(f"Loaded {len(notion_docs)} markdown files from {docs_dir}")


Loaded 6 markdown files from rag_exp\notion_docs


In [3]:
'''notion_docs = {
    key: value for key, value in notion_docs.items() if "create" not in key
}
print(len(notion_docs))'''

'notion_docs = {\n    key: value for key, value in notion_docs.items() if "create" not in key\n}\nprint(len(notion_docs))'

In [4]:
testing_doc = notion_docs["block_reference"] 

In [5]:
testing_doc = "\n\n\n".join(notion_docs.values())

In [6]:
import re
from chonkie import MarkdownChef, CodeChunker, RecursiveChunker

# Method 1: MarkdownChef
print("=" * 60)
print("Method 1: MarkdownChef")
print("=" * 60)
md_chef = MarkdownChef()
md_doc = md_chef.parse(testing_doc)
print(f"Document type: {type(md_doc)}")
print(f"Total chunks: {len(md_doc.chunks)}")
if md_doc.chunks:
    print(f"\nFirst chunk:\n{md_doc.chunks[0]}\n")

# Method 2: CodeChunker
print("=" * 60)
print("Method 2: CodeChunker")
print("=" * 60)
code_chunker = CodeChunker()
print(f"CodeChunker methods: {[m for m in dir(code_chunker) if not m.startswith('_')]}")
code_chunks = code_chunker.chunk(testing_doc)
print(f"Total chunks: {len(code_chunks)}")
if code_chunks:
    print(f"\nFirst chunk:\n{code_chunks[0]}\n")

# Method 3: RecursiveChunker
print("=" * 60)
print("Method 3: RecursiveChunker")
print("=" * 60)
recursive_chunker = RecursiveChunker()
recursive_chunks = recursive_chunker.chunk(testing_doc)
print(f"Total chunks: {len(recursive_chunks)}")
if recursive_chunks:
    print(f"\nFirst chunk:\n{recursive_chunks[0]}\n")


Method 1: MarkdownChef
Document type: <class 'chonkie.types.markdown.MarkdownDocument'>
Total chunks: 73

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation Index
> Fetch the complete documentation index at: https://developers.notion.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Block

> A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).

For example, the following block object represents a `Heading 2` in the Notion U

d:\DevTools\Python313\Lib\site-packages\chonkie\chunker\code.py:76: UserWarning: The language is set to `auto`. This would adversely affect the performance of the chunker. Consider setting the `language` parameter to a specific language to improve performance.
  warnings.warn(


Total chunks: 102

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation Index
> Fetch the complete documentation index at: https://developers.notion.com/llms.txt
> Use this file to discover all available pages before exploring further.



Method 3: RecursiveChunker
Total chunks: 272

First chunk:
---
source_url: https://developers.notion.com/reference/block.md
category: major
description: A block object represents a piece of content within Notion. The API translates the headings, toggles, paragraphs, lists, media, and more that you can interact with in the Notion UI as different [block type objects](/reference/block#block-type-objects).
---

> ## Documentation 

In [7]:
def chunk_by_tags(document: str) -> list:
    # Chunk document by matching opening/closing XML-like tags (e.g. <Note>...</Note>)
    pattern = re.compile(r'<(?P<tag>[A-Za-z0-9_\-]+)[^>]*>.*?</(?P=tag)>', re.DOTALL)

    tag_chunks = []
    start = 0
    for m in pattern.finditer(document):
        if m.start() > start:
            tag_chunks.append({"tag": None, "text": document[start:m.start()]})
        tag_chunks.append({"tag": m.group("tag"), "text": m.group(0)})
        start = m.end()
    if start < len(document):
        tag_chunks.append({"tag": None, "text": document[start:]})

    # remove empty chunks
    tag_chunks = [c for c in tag_chunks if c["text"].strip()]
    return tag_chunks

tag_chunks = chunk_by_tags(testing_doc)

print(f"Total tag-based chunks: {len(tag_chunks)}")
for i, c in enumerate(tag_chunks[:6]):
    snippet = c["text"][:200].replace("\n", "\\n")
    print(f"{i}: tag={c['tag']!r}, len={len(c['text'])}, preview={snippet!r}")

Total tag-based chunks: 121
0: tag=None, len=885, preview='---\\nsource_url: https://developers.notion.com/reference/block.md\\ncategory: major\\ndescription: A block object represents a piece of content within Notion. The API translates the headings, toggles, para'
1: tag='CodeGroup', len=1084, preview='<CodeGroup>\\n  ```json Example Block Object expandable theme={null}\\n  {\\n  \t"object": "block",\\n  \t"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",\\n  \t"parent": {\\n  \t\t"type": "page_id",\\n  \t\t"page_id": "5983'
2: tag=None, len=123, preview='\\n\\nUse the [Retrieve block children](/reference/get-block-children) endpoint to list all of the blocks on a page.\\n\\n## Keys\\n\\n'
3: tag='Note', len=277, preview='<Note>\\n  Fields marked with an \\* are available to integrations with any capabilities. Other properties require read content capabilities in order to be returned from the Notion API. Consult the [inte'
4: tag=None, len=27649, preview='\\n\\n| Field              | Typ

In [8]:
# Global tracking variables for inspection
global_stats = {
    "total_code": 0,
    "total_images": 0,
    "total_tables": 0,
    "total_metadata": 0,
}

def process_tag_chunks(tag_chunks: list) -> list:
    """
    Process tag_chunks through MarkdownChef to extract nested chunks, tables, code, and images.
    Removes auxiliary fields like 'chunk_type' and 'source' from the produced chunks.
    """
    global global_stats

    processed_chunks = []
    md_chef = MarkdownChef()

    for tag_chunk in tag_chunks:
        original_tag = tag_chunk["tag"]
        text = tag_chunk["text"]

        if original_tag is not None:
            processed_chunks.append({
                "tag": original_tag,
                "text": text,
            })
            continue

        # Parse with MarkdownChef
        try:
            md_doc = md_chef.parse(text)
        except Exception:
            # If parsing fails, keep as single text chunk
            print(f"MarkdownChef parsing failed for chunk with tag={original_tag!r}, treating as plain text.")
            processed_chunks.append({
                "tag": original_tag,
                "text": text,
            })
            continue

        # Extract and track metadata
        if md_doc.metadata:
            global_stats["total_metadata"] += len(md_doc.metadata) if isinstance(md_doc.metadata, list) else 1

        # Process parsed chunks from MarkdownChef
        for chunk in md_doc.chunks:
            processed_chunks.append({
                "tag": "text",
                "text": chunk.text,
            })

        # Extract and track tables
        if md_doc.tables:
            global_stats["total_tables"] += len(md_doc.tables)
            for i, table in enumerate(md_doc.tables):
                processed_chunks.append({
                    "tag": "table",
                    "text": table.content,  # Convert table object to string
                    "table_index": i
                })

        # Extract and track code
        if md_doc.code:
            global_stats["total_code"] += len(md_doc.code)
            for i, code in enumerate(md_doc.code):
                processed_chunks.append({
                    "tag": 'CodeGroup',
                    "text": code.content,  # Convert code object to string
                    "code_index": i
                })

        # Extract and track images
        if md_doc.images:
            global_stats["total_images"] += len(md_doc.images)
            for i, image in enumerate(md_doc.images):
                processed_chunks.append({
                    "tag": "image",
                    "text": image.content,  # Convert image object to string
                    "image_index": i
                })

    return processed_chunks

# Apply function to tag_chunks
expanded_chunks = process_tag_chunks(tag_chunks)

print(f"Initial tag chunks: {len(tag_chunks)}")
print(f"Expanded chunks after MarkdownChef processing: {len(expanded_chunks)}")
print(f"\nGlobal statistics:")
print(f"  Total code blocks: {global_stats['total_code']}")
print(f"  Total tables: {global_stats['total_tables']}")
print(f"  Total images: {global_stats['total_images']}")
print(f"  Total metadata: {global_stats['total_metadata']}")

print(f"\nFirst 10 expanded chunks:")
for i, chunk in enumerate(expanded_chunks[:10]):
    snippet = chunk["text"][:100].replace("\n", "\\n")
    type_label = chunk['tag']
    print(f"{i}: tag={chunk['tag']!r}, type={type_label!r}, len={len(chunk['text'])}, preview={snippet!r}")

unique_tags = set(c['tag'] for c in expanded_chunks)
print(f"\nUnique tags in expanded chunks: {unique_tags}")

# find None tags
print(f"\nChunks with None tag:")
for i, chunk in enumerate(expanded_chunks):
    if chunk['tag'] is None:
        snippet = chunk["text"][:100].replace("\n", "\\n")
        print(f"{i}: tag={chunk['tag']!r}, len={len(chunk['text'])}, preview={snippet!r}")

Initial tag chunks: 121
Expanded chunks after MarkdownChef processing: 156

Global statistics:
  Total code blocks: 3
  Total tables: 29
  Total images: 0
  Total metadata: 0

First 10 expanded chunks:
0: tag='text', type='text', len=885, preview='---\\nsource_url: https://developers.notion.com/reference/block.md\\ncategory: major\\ndescription: A bloc'
1: tag='CodeGroup', type='CodeGroup', len=1084, preview='<CodeGroup>\\n  ```json Example Block Object expandable theme={null}\\n  {\\n  \t"object": "block",\\n  \t"id"'
2: tag='text', type='text', len=123, preview='\\n\\nUse the [Retrieve block children](/reference/get-block-children) endpoint to list all of the block'
3: tag='Note', type='Note', len=277, preview='<Note>\\n  Fields marked with an \\* are available to integrations with any capabilities. Other propert'
4: tag='text', type='text', len=1103, preview='\\n#### Block types that support child blocks\\n\\nSome block types contain nested blocks. The following b'
5: tag='table', t

#### Parsing big yaml, tables, code

In [9]:
for i, c in enumerate(expanded_chunks):
    tag = c.get("tag")
    text = c.get("text", "") or ""
    if len(text) > 30000:
        break

In [10]:
code_chunker = CodeChunker()
r = code_chunker(c['text'][2:])
len(r)

1

In [11]:
c['text'][43:100]

'openapi: 3.1.0\ninfo:\n  title: Notion API\n  version: 1.0.0'

In [12]:
c['text'][-43:]

'uth:\n      type: http\n      scheme: bearer\n'

##### Manual + regex

In [13]:
# find the first "openapi" occurrence, trim everything before its line, then load YAML
text_fragment = c["text"] or ""

m = re.search(r"^\s*openapi\s*:", text_fragment, flags=re.IGNORECASE | re.MULTILINE)
if not m:
    m = re.search(r"openapi", text_fragment, flags=re.IGNORECASE)

if m:
    # start from the beginning of the line that contains the match
    line_start = text_fragment.rfind("\n", 0, m.start()) + 1
    yaml_text = text_fragment[line_start:]
else:
    yaml_text = text_fragment  # fallback: try to load whole text

try:
    parsed_yaml = yaml.load(yaml_text, Loader=yaml.FullLoader)
    print("YAML loaded, top-level keys:", list(parsed_yaml.keys()) if isinstance(parsed_yaml, dict) else type(parsed_yaml))
except Exception as e:
    print("YAML load failed:", e)
    parsed_yaml = None

YAML loaded, top-level keys: ['openapi', 'info', 'servers', 'security', 'tags', 'paths', 'components']


#### By line

In [14]:
import importlib

# Use PyYAML's safe_load from sys.modules (avoid overwriting existing `yaml` variable)
pyyaml = importlib.import_module("yaml")

def split_zero_indent(text: str) -> list:
    """Split text by lines starting at column 0 that contain a colon (top-level YAML keys)."""
    header_re = re.compile(r'^[^\s].*?:')
    lines = text.splitlines(True)
    blocks = []
    current = ""
    for line in lines:
        if header_re.match(line) and current.strip():
            blocks.append(current)
            current = line
        else:
            current += line
    if current:
        blocks.append(current)
    return blocks

# Use existing yaml_text variable from the notebook
blocks = split_zero_indent(yaml_text)

parsed_results = []
for i, blk in enumerate(blocks, start=1):
    try:
        parsed = pyyaml.safe_load(blk)
        parsed_results.append({"idx": i, "text": blk, "parsed": parsed, "error": None})
    except Exception as e:
        parsed_results.append({"idx": i, "text": blk, "parsed": None, "error": str(e)})

total = len(parsed_results)
parsed_count = sum(1 for r in parsed_results if r["parsed"] is not None)
failed_indices = [r["idx"] for r in parsed_results if r["parsed"] is None]

print(f"Blocks: {total}, Parsed: {parsed_count}, Failed: {len(failed_indices)} ({(parsed_count/total)*100:.1f}% rescued)")

if failed_indices:
    print("Failed indices (sample):", failed_indices[:10])
    first_fail = next(r for r in parsed_results if r["parsed"] is None)
    preview = first_fail["text"][:400].replace("\n", "\\n")
    print(f"First failed idx={first_fail['idx']}, error={first_fail['error']}")
    print("Preview:", preview)

# Expose for downstream cells
rescued_blocks = [r for r in parsed_results if r["parsed"] is not None]
flagged_blocks = [r for r in parsed_results if r["parsed"] is None]

Blocks: 7, Parsed: 7, Failed: 0 (100.0% rescued)


#### Parsing jsons

In [15]:
import json

json_blocks = [json.dumps(r["parsed"], indent=2) for r in rescued_blocks]
json_chunker = CodeChunker(language="json")
json_chunks = [json_chunker.chunk(block) for block in json_blocks]
print(f"Total JSON blocks: {len(json_blocks)}")
print(f"Total JSON chunks: {len(json_chunks)}")

Total JSON blocks: 7
Total JSON chunks: 7


In [16]:
from langchain_text_splitters import RecursiveJsonSplitter

json_splitter = RecursiveJsonSplitter(max_chunk_size=1000)

2026-03-15 23:05:36 | INFO     | datasets | TensorFlow version 2.20.0 available.
2026-03-15 23:05:45 | WARNING  | tensorflow | From d:\DevTools\Python313\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



In [17]:
# Split per rescued block, then flatten and improved prints
json_chunks_by_block = [json_splitter.split_text(r["parsed"]) for r in rescued_blocks]
total_blocks = len(json_chunks_by_block)
json_chunks = [chunk for chunks in json_chunks_by_block for chunk in chunks]

print(f"Blocks processed: {total_blocks}")
print(f"Total JSON chunks after RecursiveJsonSplitter: {len(json_chunks)}")

for i, chunk in enumerate(json_chunks[:10]):
    preview = str(chunk)[:200].replace("\n", "\\n")
    print(f"  [{i}] preview={preview}")

Blocks processed: 7
Total JSON chunks after RecursiveJsonSplitter: 92
  [0] preview={"openapi": "3.1.0"}
  [1] preview={"info": {"title": "Notion API", "version": "1.0.0", "termsOfService": "https://notion.notion.site/Terms-and-Privacy-28ffdd083dc3473e9c2da6ec011b58ac"}}
  [2] preview={"servers": [{"url": "https://api.notion.com"}]}
  [3] preview={"security": [{"bearerAuth": []}]}
  [4] preview={"tags": [{"name": "Databases", "description": "Database endpoints"}, {"name": "Data sources", "description": "Data source endpoints"}, {"name": "Pages", "description": "Page endpoints"}, {"name": "Bl
  [5] preview={"paths": {"/v1/blocks/{block_id}/children": {"patch": {"tags": ["Blocks"], "summary": "Append block children", "operationId": "patch-block-children", "parameters": [{"name": "block_id", "in": "path",
  [6] preview={"paths": {"/v1/blocks/{block_id}/children": {"patch": {"responses": {"200": {"content": {"application/json": {"schema": {"properties": {"type": {"type": "string", "const":

In [18]:
json_chunks[1]

'{"info": {"title": "Notion API", "version": "1.0.0", "termsOfService": "https://notion.notion.site/Terms-and-Privacy-28ffdd083dc3473e9c2da6ec011b58ac"}}'

In [19]:
json_splitter.split_text(rescued_blocks[0]["parsed"])

['{"openapi": "3.1.0"}']

#### Ruamel

In [20]:
from ruamel.yaml import YAML
from ruamel.yaml.error import YAMLError

# Mini example: tolerant YAML parsing with ruamel.yaml and helpful error location
# Requires: pip install ruamel.yaml

# Use existing variables (c, re) from the notebook
text_fragment = c.get("text", "") or ""
yaml_text = text_fragment  # In practice, you might want to trim to the relevant section as shown before

yaml = YAML()
try:
    parsed = yaml.load(yaml_text)
    print("YAML loaded successfully; top-level type:", type(parsed))
except YAMLError as e:
    print("ruamel.yaml parse error:", e)
    mark = getattr(e, "problem_mark", None) or getattr(e, "context_mark", None)
    if mark:
        ln, col = mark.line, mark.column
        print(f"Error location -> line {ln+1}, column {col+1}")
        lines = yaml_text.splitlines()
        start = max(0, ln - 3)
        end = min(len(lines), ln + 4)
        for i in range(start, end):
            prefix = ">>" if i == ln else "  "
            print(f"{prefix} {i+1}: {lines[i]}")
    else:
        print("No line/column info available from the error.")

ruamel.yaml parse error: while scanning for the next token
found character '`' that cannot start any token
  in "<unicode string>", line 1, column 1:
    `yaml patch /v1/blocks/{block_id ... 
    ^ (line: 1)
Error location -> line 1, column 1
>> 1: `yaml patch /v1/blocks/{block_id}/children
   2: openapi: 3.1.0
   3: info:
   4:   title: Notion API


In [21]:
r = recursive_chunker.chunk(text)
r[0]

Chunk(text='`yaml patch /v1/blocks/{block_id}/children
openapi: 3.1.0
info:
  title: Notion API
  version: 1.0.0
  termsOfService: >-
    https://notion.notion.site/Terms-and-Privacy-28ffdd083dc3473e9c2da6ec011b58ac
servers:
  - url: https://api.notion.com
security:
  - bearerAuth: []
tags:
  - name: Databases
    description: Database endpoints
  - name: Data sources
    description: Data source endpoints
  - name: Pages
    description: Page endpoints
  - name: Blocks
    description: Block endpoints
  - name: Comments
    description: Comment endpoints
  - name: File uploads
    description: File upload endpoints
  - name: OAuth
    description: OAuth endpoints (basic authentication)
  - name: Users
    description: User endpoints
  - name: Search
    description: Search endpoints
paths:
  /v1/blocks/{block_id}/children:
    patch:
      tags:
        - Blocks
      summary: Append block children
      operationId: patch-block-children
      parameters:
        - name: block_id
    

In [22]:
# Use existing chunkers (code_chunker, recursive_chunker). Do not re-import.
CHUNK_SIZE = getattr(recursive_chunker, "chunk_size", 2048)

recursive_chunker = RecursiveChunker(chunk_size=CHUNK_SIZE)
code_chunker = CodeChunker(language="yaml")

code_group_inspect = []
recursive_inspect = []
table_inspect = []
other_inspect = []

for i, c in enumerate(expanded_chunks):
    tag = c.get("tag")
    text = c.get("text", "") or ""
    if len(text) > 5000:
        print(f"Long chunk at idx={i} with tag={tag!r}, len={len(text)}")
        if tag == "CodeGroup":
            code_subs = code_chunker.chunk(text)
            # code_chunker returns strings; keep first few chars for preview
            preview = code_subs[0][:500] if code_subs else ""
            code_group_inspect.append({"idx": i, "n_subchunks": len(code_subs), "preview": preview})
        elif tag == "text" and len(text) > CHUNK_SIZE:
            sub_chunks = recursive_chunker.chunk(text)
            # RecursiveChunker returns Chunk objects with .text
            n = len(sub_chunks)
            first = getattr(sub_chunks[0], "text", str(sub_chunks[0]))[:500] if n else ""
            recursive_inspect.append({"idx": i, "n_subchunks": n, "first_preview": first})
        elif tag == "table":
            rows = [r for r in text.splitlines() if r.strip()]
            table_inspect.append({"idx": i, "n_rows": len(rows), "preview": "\n".join(rows[:5])})
        else:
            other_inspect.append({"idx": i, "tag": tag, "len": len(text), "preview": text[:200]})

# Quick summary prints
print(f"CHUNK_SIZE={CHUNK_SIZE}")
print(f"CodeGroup chunks found: {len(code_group_inspect)}")
for item in code_group_inspect[:5]:
    print(f"  idx={item['idx']}, subchunks={item['n_subchunks']}, preview_len={len(item['preview'])}")

print(f"Long text chunks (split by RecursiveChunker): {len(recursive_inspect)}")
for item in recursive_inspect[:5]:
    print(f"  idx={item['idx']}, subchunks={item['n_subchunks']}, first_preview_len={len(item['first_preview'])}")

print(f"Table chunks: {len(table_inspect)}")
for item in table_inspect[:5]:
    print(f"  idx={item['idx']}, rows={item['n_rows']}, preview_lines=\n{item['preview']}")

print(f"Other chunks sampled: {len(other_inspect)}")
for item in other_inspect[:5]:
    print(f"  idx={item['idx']}, tag={item['tag']}, len={item['len']}, preview={item['preview']!r}")

Long chunk at idx=5 with tag='table', len=26544
Long chunk at idx=137 with tag='CodeGroup', len=154748
Long chunk at idx=138 with tag='CodeGroup', len=81024
Long chunk at idx=144 with tag='CodeGroup', len=149871
Long chunk at idx=148 with tag='table', len=9760
Long chunk at idx=155 with tag='table', len=14608
CHUNK_SIZE=2048
CodeGroup chunks found: 3
  idx=137, subchunks=1, preview_len=500
  idx=138, subchunks=1, preview_len=500
  idx=144, subchunks=1, preview_len=500
Long text chunks (split by RecursiveChunker): 0
Table chunks: 3
  idx=5, rows=14, preview_lines=
| Field              | Type                                                                    | Description                                                                                                                                                                                                                                                                                                                                  

In [28]:
# find a table chunk > 20000 chars and split with RecursiveChunker(chunk_size=1000)
CHUNK_SIZE = 5000

rc = RecursiveChunker(chunk_size=CHUNK_SIZE)

match = next(((i, c) for i, c in enumerate(expanded_chunks)
              if c.get("tag") == "table" and len(c.get("text", "")) > 20000), None)

if match is None:
    print("No table chunk >20000 chars found.")
else:
    idx, chunk = match
    text = chunk.get("text", "")
    print(f"Found table chunk at idx={idx}, len={len(text)}. Splitting with CHUNK_SIZE={CHUNK_SIZE}...")
    parts = rc.chunk(text)
    print(f"Produced {len(parts)} sub-chunks. Showing first 5:")
    for j, p in enumerate(parts[:5]):
        p_text = getattr(p, "text", p)
        preview = p_text.replace("\n", " ").replace(" ", "").replace("-", "")[:100]
        print(f"#{j}: len={len(p_text)}, preview={preview}...")

Found table chunk at idx=5, len=26544. Splitting with CHUNK_SIZE=5000...
Produced 7 sub-chunks. Showing first 5:
#0: len=3792, preview=|Field|Type|Description|Examplevalue||:|:|:|:|...
#1: len=3792, preview=|`object`\*|`string`|Always`"block"`.|`"block"`||`id`\*|`string`(UUIDv4)|Identifierfortheblock.|`"7a...
#2: len=3792, preview=|`parent`|`object`|Informationabouttheblock'sparent.See[Parentobject](/reference/parentobject).|`{"t...
#3: len=3792, preview=|`created_time`|`string`([ISO8601datetime](https://en.wikipedia.org/wiki/ISO_8601))|Dateandtimewhent...
#4: len=3792, preview=|`last_edited_time`|`string`([ISO8601datetime](https://en.wikipedia.org/wiki/ISO_8601))|Dateandtimew...


In [27]:
match

(5,
 {'tag': 'table',
  'text': '| Field              | Type                                                                    | Description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

Unique tags in expanded chunks: {'Check', 'Note', 'Danger', 'text', 'Tip', 'Frame', 'Steps', 'Info', 'Warning', 'table', 'CodeGroup'}


#### Fixing stuff

In [40]:
import importlib
import json
import re

pyyaml = importlib.import_module("yaml")

CHUNK_SIZE = 1024
recursive_chunker_1024 = RecursiveChunker(chunk_size=CHUNK_SIZE)
json_chunker = CodeChunker(language="json")

def _as_text_parts(parts):
    out = []
    for part in parts or []:
        out.append(getattr(part, "text", str(part)))
    return out

def extract_yaml_candidates(text: str) -> list[str]:
    candidates = []

    # Prefer fenced YAML blocks when available.
    fenced = re.findall(r"```(?:yaml|yml)\s*\n(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    for block in fenced:
        if block.strip():
            candidates.append(block.strip())

    # Fallback: capture from first likely YAML key (common in OpenAPI/spec docs).
    start_match = re.search(r"(?im)^\s*(openapi|swagger|info|paths|components|servers)\s*:\s*", text)
    if start_match:
        line_start = text.rfind("\n", 0, start_match.start()) + 1
        tail = text[line_start:].strip()
        if tail:
            candidates.append(tail)

    # Final fallback: if text itself looks YAML-ish, include it.
    if re.search(r"(?m)^\s*[^#\s][^:\n]*:\s*", text):
        stripped = text.strip()
        if stripped:
            candidates.append(stripped)

    # Deduplicate while preserving order.
    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            unique.append(c)
            seen.add(c)
    return unique

def parse_yaml_elements(candidates: list[str]) -> list:
    parsed = []

    def try_parse(fragment: str):
        try:
            obj = pyyaml.safe_load(fragment)
        except Exception:
            return None
        if obj is None:
            return None
        if isinstance(obj, (dict, list)):
            return obj
        return None

    for candidate in candidates:
        # 1) Try whole candidate first.
        obj = try_parse(candidate)
        if obj is not None:
            parsed.append(obj)
            continue

        # 2) Try YAML document splits.
        docs = [d for d in re.split(r"\n---\s*\n", candidate) if d.strip()]
        for doc in docs:
            obj = try_parse(doc)
            if obj is not None:
                parsed.append(obj)

        # 3) Try top-level key blocks (safe salvage mode).
        lines = candidate.splitlines(True)
        blocks = []
        current = ""
        header_re = re.compile(r"^[^\s].*?:")
        for line in lines:
            if header_re.match(line) and current.strip():
                blocks.append(current)
                current = line
            else:
                current += line
        if current.strip():
            blocks.append(current)

        for block in blocks:
            obj = try_parse(block)
            if obj is not None:
                parsed.append(obj)

    return parsed

def split_table_rows(table_text: str) -> list[str]:
    rows = []
    for raw_line in table_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if "|" not in line:
            continue
        # Skip markdown separator rows like |---|---:|
        compact = line.replace("|", "").replace(" ", "")
        if compact and all(ch in "-:" for ch in compact):
            continue
        rows.append(line)
    return rows if rows else [table_text]

def apply_requested_chunking(chunks: list, chunk_size: int = 1024) -> list:
    stage_1 = []

    for chunk in chunks:
        text = (chunk.get("text") or "").strip()
        if not text:
            continue

        tag = chunk.get("tag")

        # 1) Tables: one row = one chunk.
        if tag == "table":
            for row_idx, row_text in enumerate(split_table_rows(text)):
                row_chunk = dict(chunk)
                row_chunk["tag"] = "table_row"
                row_chunk["text"] = row_text
                row_chunk["row_index"] = row_idx
                stage_1.append(row_chunk)
            continue

        # 2) YAML flow: regex extraction -> safe parse salvage -> JSON chunking.
        yaml_candidates = extract_yaml_candidates(text)
        parsed_yaml = parse_yaml_elements(yaml_candidates)

        if parsed_yaml:
            for element_idx, element in enumerate(parsed_yaml):
                json_text = json.dumps(element, ensure_ascii=True, indent=2)
                json_parts = _as_text_parts(json_chunker.chunk(json_text)) or [json_text]
                for json_idx, part_text in enumerate(json_parts):
                    stage_1.append({
                        "tag": "yaml_json",
                        "text": part_text,
                        "source_tag": tag,
                        "yaml_element_index": element_idx,
                        "json_chunk_index": json_idx,
                    })
            continue

        stage_1.append(dict(chunk))

    # 3) Recursive chunker for all chunks with text.
    rc = RecursiveChunker(chunk_size=chunk_size)
    final_chunks = []
    for chunk in stage_1:
        text = chunk.get("text", "") or ""
        if not text:
            continue

        parts = _as_text_parts(rc.chunk(text)) or [text]
        total = len(parts)
        for part_idx, part_text in enumerate(parts):
            segmented = dict(chunk)
            segmented["text"] = part_text
            segmented["segment_index"] = part_idx
            segmented["segment_total"] = total
            final_chunks.append(segmented)

    return final_chunks

fixed_chunks = apply_requested_chunking(expanded_chunks, chunk_size=CHUNK_SIZE)
expanded_chunks = fixed_chunks

print(f"Applied fixes. Total chunks now: {len(expanded_chunks)}")
print(f"table_row chunks: {sum(1 for c in expanded_chunks if c.get('tag') == 'table_row')}")
print(f"yaml_json chunks: {sum(1 for c in expanded_chunks if c.get('tag') == 'yaml_json')}")
print(f"Chunks with segmentation metadata: {sum(1 for c in expanded_chunks if 'segment_index' in c)}")

Applied fixes. Total chunks now: 1709
table_row chunks: 168
yaml_json chunks: 1444
Chunks with segmentation metadata: 1709


### Continue

In [34]:
def build_summary_prompt(prior_context, snippet):
    prompt = f"""
You are generating retrieval-oriented technical summaries for markdown chunks.

Context:
<context>{prior_context}</context>

Snippet:
<snippet>{snippet}</snippet>

Instructions:
- Write a compact, factual summary of the snippet content only.
- Use dry technical language (e.g., "implemented", "used", "defines", "returns", "contains").
- Remove narrative filler and meta commentary.
- Do NOT use phrases like:
  - "this snippet is about ..."
  - "the text describes ..."
  - "the section discusses ..."
- Do not repeat the prompt or mention summarization steps.
- Max length: 70 words.
- Preserve high-signal details: field names, API objects, constraints, types, enums, parameters, and behavior.

Type-specific rules:
- Table: report schema/columns and key values or constraints.
- Code: report what is implemented, inputs/outputs, and core logic.
- Image: report concrete visible/embedded technical content only.
- Metadata: report explicit keys and values.
"""
    return prompt.strip()

In [35]:
import random
from src.all_functionality import async_chat_wrapper

# Array of target models to iterate over
MODELS_TO_TEST = ["gemma4", "gemma12", "gemma27"]

async def ask_llm(prompt):
    # Randomly select a model from the list
    selected_model = random.choice(MODELS_TO_TEST)
    
    # Placeholder for LLM interaction
    return await async_chat_wrapper(
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        model_size=selected_model
    )

In [36]:
import asyncio
from typing import Dict, Any

async def summarize_chunks_with_concurrency(chunks: list, max_concurrent: int = 10) -> list:
    """
    For every table/CodeGroup chunk, collect prior context (previous text chunk)
    and ask LLM with summarization prompt. Add "summary" field to these chunks.
    Process in parallel with max_concurrent limit.
    """
    
    # Create a semaphore to limit concurrent requests
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def summarize_single(chunk_idx: int, chunk: Dict[str, Any]) -> None:
        """Summarize a single chunk if it's a table or CodeGroup"""
        if chunk["tag"] not in ["table", "CodeGroup"]:
            return
        
        # Find prior text chunk
        prior_context = ""
        for i in range(chunk_idx - 1, -1, -1):
            if chunks[i]["tag"] == "text":
                prior_context = chunks[i]["text"]
                break
        
        if not prior_context:
            prior_context = "(No prior text context found)"
        
        # Build prompt
        prompt = build_summary_prompt(prior_context, chunk["text"])
        
        # Call LLM with semaphore
        async with semaphore:
            try:
                response = await ask_llm(prompt)
                chunk["summary"] = response
                print(f"✓ Summarized chunk {chunk_idx} (tag: {chunk['tag']}, len: {len(chunk['text'])})")
            except Exception as e:
                chunk["summary"] = f"Error: {str(e)}"
                print(f"✗ Error summarizing chunk {chunk_idx}: {str(e)}")
    
    # Create tasks for all chunks
    tasks = [summarize_single(i, chunk) for i, chunk in enumerate(chunks)]
    
    # Run all tasks concurrently with semaphore limit
    await asyncio.gather(*tasks)
    
    return chunks

# Run summarization
print("Starting parallel summarization of table/CodeGroup chunks...")
print(f"Processing {len([c for c in expanded_chunks if c['tag'] in ['table', 'CodeGroup']])} chunks with max_concurrent=10")
print()

expanded_chunks = await summarize_chunks_with_concurrency(expanded_chunks, max_concurrent=10)

print("\nSummarization complete!")
print(f"Chunks with summaries: {len([c for c in expanded_chunks if 'summary' in c])}")

# Show samples
print("\nSample summaries:")
for i, chunk in enumerate(expanded_chunks):
    if "summary" in chunk:
        print(f"\n[Chunk {i}] Tag: {chunk['tag']}")
        print(f"Preview: {chunk['text'][:150].replace(chr(10), ' ')}...")
        print(f"Summary: {chunk['summary'][:200]}")
        if i >= 2:  # Show first 3 with summaries
            break


Starting parallel summarization of table/CodeGroup chunks...
Processing 45 chunks with max_concurrent=10



2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 83 (tag: CodeGroup, len: 237)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 73 (tag: CodeGroup, len: 521)
[async_chat_wrapper] Model: gemma-3-4b-it, finish_reason: stop
✓ Summarized chunk 65 (tag: CodeGroup, len: 260)


2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-15 23:57:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop
✓ Summarized chunk 95 (tag: CodeGroup, len: 360)
[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop
✓ Summarized chunk 60 (tag: CodeGroup, len: 274)


...

[async_chat_wrapper] Model: gemma-3-12b-it, finish_reason: stop
✓ Summarized chunk 228 (tag: CodeGroup, len: 446)


2026-03-16 00:00:02 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-12b-it, finish_reason: stop
✓ Summarized chunk 240 (tag: CodeGroup, len: 277)


2026-03-16 00:00:31 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-12b-it, finish_reason: stop
✓ Summarized chunk 129 (tag: CodeGroup, len: 400)

Summarization complete!
Chunks with summaries: 105

Sample summaries:

[Chunk 4] Tag: CodeGroup
Preview: <CodeGroup>   ```json Example Block Object expandable theme={null}   {   	"object": "block",   	"id": "c02fc1d3-db8b-45c5-a222-27595b15aea7",   	"pare...
Summary: The snippet represents a block object with an ID of `c02fc1d3-db8b-45c5-a222-27595b15aea7`. It’s a child of page ID `59833787-2cf9-4fdf-8782-e53db20768a5`. The block is a heading level 2, containing t


In [44]:
for i in range(len(expanded_chunks)):
    c = expanded_chunks[i]
    if c.get("tag") != "CodeGroup":
        c['summary'] = None

In [46]:
def build_final_tag(chunks):
    new_chunks = []
    for chunk in chunks:
        if chunk.get("summary"):
            name = "summary"
        else:
            name = "text"
            
        chunk["final"] = f"<{chunk['tag']}>{chunk[name]}</{chunk['tag']}>"
        new_chunks.append(chunk)
    return new_chunks

final_chunks = build_final_tag(expanded_chunks)

In [47]:
# Minimal inspections on final_chunks
print("=" * 60)
print("Final Chunks Inspection")
print("=" * 60)

print(f"\nTotal final chunks: {len(final_chunks)}")

# Count chunks by tag
tag_counts = {}
for chunk in final_chunks:
    tag = chunk.get("tag")
    tag_counts[tag] = tag_counts.get(tag, 0) + 1

print(f"\nChunks by tag:")
for tag, count in sorted(tag_counts.items()):
    print(f"  {tag}: {count}")

# Check for summary presence
chunks_with_summary = len([c for c in final_chunks if "summary" in c])
print(f"\nChunks with summaries: {chunks_with_summary}")

# Length statistics
lengths = [len(chunk.get("final", "")) for chunk in final_chunks]
print(f"\nFinal field length statistics:")
print(f"  Min: {min(lengths)}")
print(f"  Max: {max(lengths)}")
print(f"  Mean: {sum(lengths) / len(lengths):.0f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]}")

# Show sample of each tag type
print(f"\nSample chunks by tag:")
seen_tags = set()
for i, chunk in enumerate(final_chunks):
    tag = chunk.get("tag")
    if tag not in seen_tags:
        print(f"\n[{tag}] Chunk {i} (len: {len(chunk['final'])})")
        print(f"  Preview: {chunk['final'][:150].replace(chr(10), ' ')}...")
        seen_tags.add(tag)

Final Chunks Inspection

Total final chunks: 1709

Chunks by tag:
  Check: 1
  CodeGroup: 45
  Danger: 1
  Frame: 1
  Info: 2
  Note: 13
  Steps: 2
  Tip: 1
  table_row: 168
  text: 23
  yaml_json: 1444

Chunks with summaries: 1709

Final field length statistics:
  Min: 29
  Max: 1047
  Mean: 729
  Median: 884

Sample chunks by tag:

[yaml_json] Chunk 0 (len: 389)
  Preview: <yaml_json>{   "source_url": "https://developers.notion.com/reference/block.md",   "category": "major",   "description": "A block object represents a ...

[CodeGroup] Chunk 4 (len: 345)
  Preview: <CodeGroup>The snippet represents a block object with an ID of `c02fc1d3-db8b-45c5-a222-27595b15aea7`. It’s a child of page ID `59833787-2cf9-4fdf-878...

[text] Chunk 6 (len: 136)
  Preview: <text>  Use the [Retrieve block children](/reference/get-block-children) endpoint to list all of the blocks on a page.  ## Keys  </text>...

[Note] Chunk 7 (len: 290)
  Preview: <Note><Note>   Fields marked with an \* are available t

apply recursive chunker for all text chunks with a UPPERCASED local varibles CHUNK_SIZE

### QDRANT CLIENT

In [48]:
from qdrant_client.models import Distance, HnswConfigDiff, PointStruct
import secrets
import string

# ──────────────────────────────────────────────────────────────────────────────
# 1. Setup Qdrant client
# ──────────────────────────────────────────────────────────────────────────────

qdrant_client = QdrantClient(path="./rag_exp/qdrant_data", prefer_grpc=True)
COLLECTION_NAME = "notion_chunks"

# ──────────────────────────────────────────────────────────────────────────────
# 2. Utility function: Generate 8-character UUID identifiers
# ──────────────────────────────────────────────────────────────────────────────

def generate_short_uuid(length: int = 8) -> str:
    """Generate a random alphanumeric identifier."""
    chars = string.ascii_letters + string.digits
    return ''.join(secrets.choice(chars) for _ in range(length))

# Create identifier mapping
chunk_ids = {}
for i, chunk in enumerate(expanded_chunks):
    chunk_ids[i] = generate_short_uuid()

print(f"Generated {len(chunk_ids)} unique identifiers")
print(f"Sample IDs: {list(chunk_ids.items())[:5]}")

Generated 1709 unique identifiers
Sample IDs: [(0, 'fQ9sBU8N'), (1, 'xg7JXhNa'), (2, 'XRGvGwYa'), (3, 'CmpfXCb9'), (4, 'oCpa3wZ7')]


In [49]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. Embed all chunks and prepare points for Qdrant
# ──────────────────────────────────────────────────────────────────────────────

embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# ──────────────────────────────────────────────────────────────────────────────
# First pass: Collect IDs and embeddings for all chunks
# ──────────────────────────────────────────────────────────────────────────────

chunk_ids = {}
chunk_embeddings = {}

for idx, chunk in enumerate(expanded_chunks):
    chunk_id_str = generate_short_uuid()
    chunk_ids[idx] = chunk_id_str
    
    # Get final field or fall back to text
    embedding_text = chunk.get("final", chunk.get("text", ""))
    
    # Generate embedding
    embedding = list(embedding_model.embed(embedding_text))[0]
    chunk_embeddings[idx] = embedding

print(f"Generated {len(chunk_ids)} unique IDs and embeddings")
print(f"Vector dimension: {len(list(chunk_embeddings.values())[0])}")

# ──────────────────────────────────────────────────────────────────────────────
# Second pass: Build neighbor relationships and create Qdrant points
# ──────────────────────────────────────────────────────────────────────────────

n_neighbors = 1
points = []

for idx, chunk in enumerate(expanded_chunks):
    chunk_id_str = chunk_ids[idx]
    chunk_type = chunk.get("tag", "text")
    raw_text = chunk.get("text", "")
    
    # Build neighbor list (indices N before and after)
    neighbor_uuids = []
    for neighbor_idx in range(max(0, idx - n_neighbors), idx):
        neighbor_uuids.append(chunk_ids[neighbor_idx])
    for neighbor_idx in range(idx + 1, min(len(expanded_chunks), idx + n_neighbors + 1)):
        neighbor_uuids.append(chunk_ids[neighbor_idx])
    
    # Create payload with all chunk info
    payload = {
        "chunk_index": idx,
        "chunk_id": chunk_id_str,
        "tag": chunk_type,
        "text": chunk.get("final"), 
        "raw_text": raw_text,
        "neighbor_ids": neighbor_uuids,
    }
    
    # Create point for Qdrant
    point = PointStruct(
        id=int(chunk_id_str.encode().hex()[:16], 16) % (2**31),  # Convert to numeric ID
        vector=chunk_embeddings[idx],
        payload=payload
    )
    points.append((chunk_id_str, point))

print(f"Built neighbor relationships for {len(points)} chunks")
print(f"Sample point ID: {points[0][0]}, neighbors: {points[0][1].payload['neighbor_ids'][:3]}")


Generated 1709 unique IDs and embeddings
Vector dimension: 384
Built neighbor relationships for 1709 chunks
Sample point ID: wJDk6yBJ, neighbors: ['zjhEs7Yw']


In [52]:

# ──────────────────────────────────────────────────────────────────────────────
# 5. Create Qdrant collection with HNSW config
# ──────────────────────────────────────────────────────────────────────────────

from qdrant_client.models import VectorParams, HnswConfigDiff, ScalarQuantizationConfig, models

vector_dim = len(points[0][1].vector)

if qdrant_client.collection_exists(COLLECTION_NAME):
    raise ValueError(f"Collection '{COLLECTION_NAME}' already exists. Please choose a different name or delete the existing collection.")

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=vector_dim,
        distance=Distance.COSINE
    ),
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=200,
        full_scan_threshold=1000
    ),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,
            quantile=0.99, # Improves accuracy by clipping outliers
            always_ram=True # Keep quantized vectors in RAM for speed
        )
    )
)

print(f"Created collection '{COLLECTION_NAME}' with HNSW config")

# Upload points to Qdrant
points_to_upload = [point for _, point in points]
qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points_to_upload
)

print(f"Uploaded {len(points_to_upload)} points to Qdrant")

# Create mapping from numeric ID back to string ID
id_mapping = {point.id: chunk_id_str for chunk_id_str, point in points}
print(f"ID mapping created: {len(id_mapping)} entries")


Created collection 'notion_chunks' with HNSW config
Uploaded 1709 points to Qdrant
ID mapping created: 1709 entries


In [53]:
def get_results(all_results):
    # Ensure combined field exists and tracking sets
    all_results.setdefault("combined", [])
    all_results.setdefault("combined_documents", [])
    seen_chunk_indices = set()
    id_to_idx = {v: k for k, v in chunk_ids.items()}

    for query_text in all_results["queries"]:
        transformed_query = query_text.lower().strip()
        query_embedding = list(embedding_model.embed(transformed_query))[0]

        search_results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_embedding,
            limit=all_results["limit_per_query"]
        ).points

        query_result = {
            "query": query_text,
            "transformed_query": transformed_query,
            "results": [],
            "neighbors": [] if all_results["use_neighbours"] else None,
        }

        for scored_point in search_results:
            payload = scored_point.payload or {}
            chunk_idx = payload.get("chunk_index")
            chunk_id = payload.get("chunk_id")
            text = payload.get("text")

            result_item = {
                "score": scored_point.score,
                "id": chunk_id,
                "chunk_index": chunk_idx,
                "tag": payload.get("tag"),
                "text": text,
                "raw_text": payload.get("raw_text"),
            }
            query_result["results"].append(result_item)

            # Add document to combined_documents and combined (unique by chunk_index)
            if chunk_idx is not None and chunk_idx not in seen_chunk_indices:
                all_results["combined_documents"].append(result_item)
                all_results["combined"].append(result_item)
                seen_chunk_indices.add(chunk_idx)

            # Collect and add neighbors (unique)
            if all_results["use_neighbours"]:
                neighbor_ids = payload.get("neighbor_ids", []) or []
                for neighbor_id in neighbor_ids:
                    # Convert string ID to numeric ID for Qdrant lookup
                    neighbor_numeric_id = int(neighbor_id.encode().hex()[:16], 16) % (2**31)
                    
                    try:
                        neighbor_points = qdrant_client.retrieve(
                            collection_name=COLLECTION_NAME,
                            ids=[neighbor_numeric_id]
                        )
                    except Exception:
                        continue
                    
                    if not neighbor_points:
                        continue
                    
                    neighbor_point = neighbor_points[0]
                    neighbor_payload = neighbor_point.payload or {}
                    neighbor_idx = neighbor_payload.get("chunk_index")
                    
                    if neighbor_idx is None or neighbor_idx in seen_chunk_indices:
                        continue

                    neighbor_item = {
                        "chunk_index": neighbor_idx,
                        "id": neighbor_id,
                        "tag": neighbor_payload.get("tag"),
                        "text": neighbor_payload.get("text", ""),
                        "raw_text": neighbor_payload.get("raw_text", ""),
                    }

                    # append to query_result neighbors and combined only if not already present
                    if not any(n["id"] == neighbor_id for n in (query_result["neighbors"] or [])):
                        query_result["neighbors"].append(neighbor_item)
                    all_results["combined"].append(neighbor_item)
                    seen_chunk_indices.add(neighbor_idx)

        all_results["query_results"].append(query_result)

    return all_results

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 6. Query Qdrant with a list of queries and return combined results
# ──────────────────────────────────────────────────────────────────────────────

# Define queries to execute
#You're a query translator. Provide me a python list of 5 variations of the following query. Apply HYde, multi query, domain knowledge, deocmposition about notion. Query: "making a to-do block object". Be conversative
queries = [
    "How to construct a 'to_do' block object in the Notion API using the children property and the rich_text schema",
    "Notion API create checkbox block JSON example",
    "notion_client blocks.children.append to_do block_id checked status",
    "Structure of a Notion block object; list of Notion block types; defining rich_text for to_do blocks",
    "Python script to programmatically add a to-do list item to a Notion page"
] + ["making a to-do block object"]
limit_per_query = 5
use_neighbours = True

all_results = {
    "queries": queries,
    "limit_per_query": limit_per_query,
    "use_neighbours": use_neighbours,
    "combined_documents": [],
    "query_results": []
}

all_results = get_results(all_results)

# print combined ids
print(list(set(r["id"] for r in all_results["combined"])))

['3X54j60I', '5mKOAq6x', 'XEhyVPYA', 'MV5AZ5MP', 'VHRHnSB0', 'JnvkN7vz', 'NdBa28gQ', 'uHjawxI3', 'zjhEs7Yw', 'Vs0oUjhZ', 'E3joOAaa', 'KpVtBCXV', 'H5aU1m8l', 'EjfRGdS7', 'DusuLOal', 'W7ZBQAvE', 'tDSenx7U', 'uE7ujFRV', 'CY7HcYUh', '2kmps7Li', 'xp1PullX', 'VuVs4olx', 'gxRTekos', 'DAXm3t9C', 'FlcbCUBj', 'WdnXTtNt', '2LBKgzgo', 'Sw1IwiVz', 'awDhCZQr', 'LDNkIiQc', '5yHg1thH', 'g5AOIMn6', 'ynawxABM', 'ImCBd2jN', 'owtffiOb', '6zors9M4', 'mYBT1Sj8', 'd85V2Kax', 'kryU7uOd', 'QbFnhz8t', 'MzRLrqSB', 'wJDk6yBJ', '48sVXpke', '6ZK5kGAv', 'RkDkHo6f', 'HrouXjQO', 'Xc6VZdQ8', 'QsG47nW1', 'Ob1UxG3e', 'J06nJZ36', '9Qo7m37d', 'xOnCcRfZ', 'lAx6iBhi', 'hfzXNAdK', 'kie0WJuw', 'MEsXQoog', 'EbkpCCgC', 'dr8XbEdc']


In [94]:
for doc in all_results["combined"]:
    print(f"Chunk index: {doc['chunk_index']}, ID: {doc['id']}, Tag: {doc['tag']}")
    if "score" in doc:
        print(f"Score: {doc['score']:.4f}")
    print(f"Text preview: {doc['text']}")

Chunk index: 228, ID: g5AOIMn6, Tag: CodeGroup
Score: 0.8060
Text preview: <CodeGroup>The JSON defines a "to_do" block object with a `type` field set to "to_do". It contains a `to_do` object with `rich_text` (containing "Finish Q3 goals"), a `checked` flag (false), a `color` ("default"), and a `children` array containing a `paragraph` object.</CodeGroup>
Chunk index: 227, ID: W7ZBQAvE, Tag: table_row
Text preview: <table_row>| `children`  | `array` of [block objects](/reference/block)         | The nested child blocks, if any, of the To do block.                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [95]:
print(len(all_results["combined"]))

58


In [96]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

model = TextCrossEncoder(model_name='jinaai/jina-reranker-v1-turbo-en')

docs = [doc["raw_text"] for doc in all_results["combined"]]

results = model.rerank(queries[0], docs)
results = list(results)

In [97]:
print(results)

[-0.26231464743614197, -1.5715861320495605, -1.5020859241485596, -0.43624967336654663, -1.3453550338745117, -3.8002872467041016, -0.045813269913196564, -3.935441255569458, -2.4257583618164062, 0.3911758363246918, -1.9214086532592773, -0.9057997465133667, 0.3816739618778229, -1.9214086532592773, -0.8779079914093018, 0.02196989208459854, -1.1983290910720825, -1.1742171049118042, -3.9648211002349854, -0.9584590196609497, -0.5070871710777283, -0.7030394077301025, -1.5836366415023804, -0.5210752487182617, -2.3731606006622314, -0.8098584413528442, -1.2973334789276123, -1.3638132810592651, -2.21250319480896, 0.15799731016159058, -1.7485194206237793, -2.8026883602142334, -0.990492582321167, -3.661376714706421, -1.5885887145996094, -2.2730207443237305, -2.849425792694092, -0.7668986320495605, -3.8002872467041016, -0.7992526888847351, -2.020392417907715, -0.41128015518188477, -0.8093559741973877, -2.020392417907715, -0.3690079152584076, -0.7844816446304321, -2.020392417907715, -0.694599032402038

In [98]:
# sort indices by value
print("QUERIES:", all_results["queries"], "\n")
sorted_results = [i for v, i in sorted([[v, i] for i, v in enumerate(list(results))], key=lambda x: x[0], reverse=True)]
for i in sorted_results[:6]:
    print(f"Score: {results[i]:.4f}, Text preview: {docs[i]}")

QUERIES: ["How to construct a 'to_do' block object in the Notion API using the children property and the rich_text schema", 'Notion API create checkbox block JSON example', 'notion_client blocks.children.append to_do block_id checked status', 'Structure of a Notion block object; list of Notion block types; defining rich_text for to_do blocks', 'Python script to programmatically add a to-do list item to a Notion page', 'making a to-do block object'] 

Score: 0.3912, Text preview: ,
        "x-codeSamples": [
          {
            "lang": "javascript",
            "label": "TypeScript SDK",
            "source": "import { Client } from \"@notionhq/client\"\n\nconst notion = new Client({ auth: process.env.NOTION_API_KEY })\n\nconst response = await notion.blocks.children.append({\n  block_id: \"c02fc1d3-db8b-45c5-a222-27595b15aea7\",\n  children: [\n    {\n      type: \"paragraph\",\n      paragraph: {\n        rich_text: [{ text: { content: \"Hello, world!\" } }]\n      }\n    }\n  ]\n